# LC 647 — Palindromic Substrings
**Day 55 | String DP / Palindromes | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Every successful (l, r) match during
an expand-around-center step IS a distinct palindrome. Count each
match — not just the widest one — and you get the total without
any extra bookkeeping.
</div>

## Official Problem Statement

Given a string `s`, return the number of palindromic substrings in it.

A string is a palindrome when it reads the same backward as forward.
A substring is a contiguous sequence of characters within the string.

**Constraints:**
- `1 <= s.length <= 1000`
- `s` consists of lowercase English letters.

## What This Is Actually Asking

Count every substring (not just the longest) that is a palindrome.
Every single character counts as a palindrome of length 1, so the
answer is always at least n.
We need to find all additional palindromes formed by pairs and longer
expansions.
The key insight is that counting is cheaper than collecting: we just
increment a counter each time expansion succeeds.
This is LC 5's cousin — same expand logic, different aggregation.

## Walk Through an Example by Hand

```
s = "aaa"   (expected: 6)

Center i=0 ('a'):
  odd : expand(0,0) -> s[0]='a'==s[0]='a'  count+1=1  ('a')
         l=-1 out of bounds -> stop
  even: expand(0,1) -> s[0]='a'==s[1]='a'  count+1=2  ('aa')
         l=-1 -> stop

Center i=1 ('a'):
  odd : expand(1,1) -> match              count+1=3  ('a')
         expand(0,2) -> s[0]='a'==s[2]='a' count+1=4 ('aaa')
         l=-1 -> stop
  even: expand(1,2) -> s[1]='a'==s[2]='a' count+1=5  ('aa')
         l=0, r=3 -> r out of bounds -> stop

Center i=2 ('a'):
  odd : expand(2,2) -> match              count+1=6  ('a')
         l=1, r=3 -> r out of bounds -> stop
  even: expand(2,3) -> r=3 out of bounds -> stop

Total = 6  CORRECT
```

## The Picture

```
s =  a  a  a
idx  0  1  2

Odd center at i=1:

         center
           |
          [a]          <- count +1  ("a")
           1

     <-- expand -->
      l=0       r=2
      s[0]='a' == s[2]='a'  -> count +1  ("aaa")

     l=-1 -> STOP

Even center between i=0 and i=1:

        |  |
       [a  a]          <- count +1  ("aa")
        0  1

     l=-1 -> STOP

Every arrow step that lands a match = +1 to total count.
```

## When To Use This Pattern

- When asked to **count** palindromic substrings (not list them),
  think **expand + increment per match**.
- When every character must be considered a center, think
  **2n-1 centers** (odd + even).
- When a DP table would work but uses O(n²) space, think
  **expand-around-center** for O(1) space.
- When the string is short (n ≤ 1000), think O(n²) time is fine.
- When you see "how many" instead of "which one", think
  **counter not tracker**.

## The Approach

Iterate over every index as an odd center and every adjacent pair
as an even center, covering all 2n-1 possible centers.
At each center, expand left and right while characters match;
every successful match increments a running counter by one.
Unlike LC 5, we do not track the longest span — we just accumulate
the total count of matching expansions.
Return the final counter.

In [ ]:
# No imports needed beyond builtins


In [ ]:
def test_harness(func):
    """
    Validate palindromic substrings count.
    Checks exact integer output against known answers.
    """
    cases = [
        ("abc",     3),
        ("aaa",     6),
        ("a",       1),
        ("aa",      3),
        ("aba",     4),
        ("abba",    6),
        ("abcba",   7),
        ("racecar", 10),
    ]

    passed = 0
    for s, expected in cases:
        result = func(s)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"{status} s={s!r} "
                f"expected={expected} got={result}"
            )
    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")


In [ ]:
def count_substrings(s: str) -> int:
    """
    Return the number of palindromic substrings in s.

    Strategy: expand-around-center.
    For each of the 2n-1 centers, expand outward while
    characters match. Each successful (l,r) pair is a
    distinct palindrome; increment count for each.

    Args:
        s: input string, 1 <= len(s) <= 1000,
           lowercase English letters only.

    Returns:
        Integer count of all palindromic substrings.
    """
    # Debug: show input
    print(f"Input: {s!r}  len={len(s)}")

    n = len(s)
    count = 0

    def expand(l, r):
        nonlocal count
        while l >= 0 and r < n and s[l] == s[r]:
            count += 1
            # Debug: show each palindrome found
            print(
                f"  +1 palindrome: s[{l}:{r+1}]="
                f"{s[l:r+1]!r}"
            )
            l -= 1
            r += 1

    for i in range(n):
        expand(i, i)      # odd length
        expand(i, i + 1)  # even length

    print(f"Total palindromic substrings: {count}")
    return count

    pass


In [ ]:
# Uncomment and run when solution is ready
# test_harness(count_substrings)


## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (check all substrings) | O(n³) | O(1) | Count valid ones |
| DP table | O(n²) | O(n²) | dp[i][j]=is palindrome |
| **Expand-around-center** | **O(n²)** | **O(1)** | Optimal in practice |
| Manacher's algorithm | O(n) | O(n) | Linear but complex |


## Real World Connection

In financial systems at Citi, counting symmetric sub-patterns in
authorization codes helps detect copy-paste or mirror-entry fraud
where operators accidentally reverse digit sequences.
On AWS, counting palindromic segments in config keys surfaces
auto-generated identifiers that have collided in a symmetric hash
space, flagging potential key conflicts in S3 partition schemes.
For data engineers, the expand-and-count pattern generalizes to
any problem where you must enumerate symmetric intervals in a
sequence — such as counting balanced bracket substrings in
serialized data payloads.
The O(1) space property makes this approach viable inside
memory-constrained Lambda functions processing large log streams.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra